In [ ]:
%%writefile file.cu
#include <iostream>
#include <cuda.h>

using namespace std;

// Kernel Function
__global__ void matrixMul(int a[][10], int b[][10], int c[][10], int n)
{
    int row = threadIdx.x;
    int col = threadIdx.y;

    int sum = 0;

    for(int k = 0; k < n; k++)
    {
        sum += a[row][k] * b[k][col];
    }

    c[row][col] = sum;
}

int main()
{
    int n;

    cout << "Enter size of matrix: ";
    cin >> n;

    int a[10][10], b[10][10], c[10][10];

    cout << "\nEnter first matrix:\n";

    for(int i = 0; i < n; i++)
    {
        for(int j = 0; j < n; j++)
        {
            cin >> a[i][j];
        }
    }

    cout << "\nEnter second matrix:\n";

    for(int i = 0; i < n; i++)
    {
        for(int j = 0; j < n; j++)
        {
            cin >> b[i][j];
        }
    }

    int (*d_a)[10], (*d_b)[10], (*d_c)[10];

    // Allocate GPU memory
    cudaMalloc((void**)&d_a, sizeof(a));
    cudaMalloc((void**)&d_b, sizeof(b));
    cudaMalloc((void**)&d_c, sizeof(c));

    // Copy matrices to GPU
    cudaMemcpy(d_a, a, sizeof(a), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, sizeof(b), cudaMemcpyHostToDevice);

    // Launch Kernel
    dim3 threads(n, n);

    matrixMul<<<1, threads>>>(d_a, d_b, d_c, n);

    // Copy result back
    cudaMemcpy(c, d_c, sizeof(c), cudaMemcpyDeviceToHost);

    cout << "\nResult Matrix:\n";

    for(int i = 0; i < n; i++)
    {
        for(int j = 0; j < n; j++)
        {
            cout << c[i][j] << " ";
        }

        cout << endl;
    }

    // Free memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing file.cu


In [ ]:
!nvcc file.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./a.out

Enter size of matrix: 3

Enter first matrix:
1 4 5
1 2 4
3 4 5

Enter second matrix:
2 3 4 
6 5 3
5 6 7

Result Matrix:
51 53 51 
34 37 38 
55 59 59 
